In [ ]:
%%html
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');

body, .jp-Notebook { background: #f0f2f5 !important; font-family: 'Inter', sans-serif !important; }

.hero {
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
    padding: 2.5rem 2rem;
    border-radius: 16px;
    margin-bottom: 1.5rem;
    color: white;
}
.hero h1 { font-size: 2rem; font-weight: 700; margin: 0 0 0.4rem 0; letter-spacing: -0.5px; }
.hero p  { font-size: 0.95rem; opacity: 0.75; margin: 0; font-weight: 300; }
.hero-badge {
    display: inline-block;
    background: rgba(255,255,255,0.15);
    border: 1px solid rgba(255,255,255,0.25);
    border-radius: 20px;
    padding: 3px 14px;
    font-size: 0.75rem;
    font-weight: 500;
    margin-bottom: 0.8rem;
    letter-spacing: 0.5px;
}

.kpi-row { display: flex; gap: 1rem; margin-bottom: 1.5rem; flex-wrap: wrap; }
.kpi-card {
    flex: 1; min-width: 180px;
    background: white;
    border: 1px solid #e8ecf0;
    border-radius: 12px;
    padding: 1.2rem 1.4rem;
    box-shadow: 0 1px 6px rgba(0,0,0,0.06);
    position: relative;
}
.kpi-label { font-size: 0.7rem; font-weight: 600; text-transform: uppercase; letter-spacing: 0.8px; color: #8492a6; margin-bottom: 0.3rem; }
.kpi-value { font-size: 1.8rem; font-weight: 700; color: #1a1a2e; line-height: 1; margin-bottom: 0.2rem; }
.kpi-sub   { font-size: 0.75rem; color: #8492a6; }
.kpi-icon  { font-size: 2rem; position: absolute; top: 1rem; right: 1.2rem; opacity: 0.12; }

.section-header { font-size: 1rem; font-weight: 600; color: #1a1a2e; margin: 0.5rem 0 0.2rem 0; }
.section-desc   { font-size: 0.82rem; color: #8492a6; margin-bottom: 0.8rem; }

.fancy-divider { border: none; height: 1px; background: linear-gradient(to right, transparent, #d8dce4, transparent); margin: 1.2rem 0; }

.widget-tab > .p-TabBar { background: #eef0f4; border-radius: 10px 10px 0 0; padding: 4px 6px 0; }
.widget-tab > .p-TabBar .p-TabBar-tab { font-family: 'Inter', sans-serif !important; font-size: 0.85rem !important; font-weight: 500; border-radius: 8px 8px 0 0; }
.widget-tab > .p-TabBar .p-TabBar-tab.p-mod-current { font-weight: 600 !important; background: white !important; }
.widget-tab > .widget-tab-contents { background: white; border-radius: 0 0 12px 12px; border: 1px solid #e8ecf0; padding: 1.2rem; box-shadow: 0 2px 8px rgba(0,0,0,0.05); }
</style>

In [ ]:
import pandas as pd
import altair as alt
import plotly.express as px
import pycountry
import ipywidgets as widgets
from IPython.display import display, HTML
from vega_datasets import data as vega_data

alt.data_transformers.disable_max_rows()

owid = pd.read_csv('owid-covid-data.csv')
owid['date'] = pd.to_datetime(owid['date'])

In [ ]:
countries_tracked = owid[
    owid['continent'].notna() & ~owid['iso_code'].str.startswith('OWID', na=True)
]['location'].nunique()

total_deaths = owid.groupby('location')['total_deaths'].max().sum()
peak_stringency = owid['stringency_index'].max()
date_range = f"{owid['date'].min().strftime('%b %Y')} \u2013 {owid['date'].max().strftime('%b %Y')}"

display(HTML(f'''
<div class="hero">
    <div class="hero-badge">\U0001f393 CS 329E \u00b7 University of Texas at Austin</div>
    <h1>\U0001f9a0 COVID-19 Policy &amp; Mortality Dashboard</h1>
    <p>Exploring how government responses shaped pandemic outcomes across 200+ countries &amp; territories</p>
</div>
<div class="kpi-row">
    <div class="kpi-card">
        <div class="kpi-icon">\U0001f30d</div>
        <div class="kpi-label">Countries Tracked</div>
        <div class="kpi-value">{countries_tracked}</div>
        <div class="kpi-sub">across 6 continents</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-icon">\U0001f4ca</div>
        <div class="kpi-label">Confirmed Deaths</div>
        <div class="kpi-value">{total_deaths/1e6:.1f}M</div>
        <div class="kpi-sub">cumulative worldwide</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-icon">\U0001f4cb</div>
        <div class="kpi-label">Peak Stringency</div>
        <div class="kpi-value">{peak_stringency:.0f}</div>
        <div class="kpi-sub">out of 100 (Oxford Index)</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-icon">\U0001f4c5</div>
        <div class="kpi-label">Date Range</div>
        <div class="kpi-value" style="font-size:1.2rem">{date_range}</div>
        <div class="kpi-sub">full pandemic timeline</div>
    </div>
</div>
<hr class="fancy-divider">
'''))

In [ ]:
# ── Tab 1: Deaths per million choropleth ─────────────────────────────────────

def to_numeric(iso3):
    try:
        return int(pycountry.countries.get(alpha_3=iso3).numeric)
    except Exception:
        return None

deaths = (
    owid.groupby(['iso_code', 'location'])['total_deaths_per_million']
    .max().reset_index()
)
deaths = deaths[deaths['total_deaths_per_million'] > 0]
deaths['id'] = deaths['iso_code'].apply(to_numeric)
deaths = deaths.dropna(subset=['id'])
deaths['id'] = deaths['id'].astype(int)

world = alt.topo_feature(vega_data.world_110m.url, 'countries')

chart1 = (
    alt.Chart(world).mark_geoshape(stroke='white', strokeWidth=0.4)
    .encode(
        color=alt.Color(
            'total_deaths_per_million:Q',
            scale=alt.Scale(scheme='redyellowgreen', reverse=True, type='log'),
            legend=alt.Legend(title='Deaths / Million (log)', format='.0f', orient='bottom-right'),
        ),
        tooltip=[
            alt.Tooltip('location:N', title='Country'),
            alt.Tooltip('total_deaths_per_million:Q', title='Deaths per Million', format=',.0f'),
        ],
    )
    .transform_lookup(
        lookup='id',
        from_=alt.LookupData(data=deaths, key='id', fields=['total_deaths_per_million', 'location']),
    )
    .project('naturalEarth1')
    .properties(width=900, height=480)
    .configure_view(strokeWidth=0)
)

# ── Tab 2: Stringency over time ───────────────────────────────────────────────

stringency_monthly = (
    owid[
        (owid['date'] >= '2020-01-01') & (owid['date'] <= '2022-12-31')
        & owid['stringency_index'].notna()
    ]
    .groupby(['location', pd.Grouper(key='date', freq='MS')])['stringency_index']
    .mean().reset_index()
)

highlight = ['United States', 'China', 'New Zealand', 'Sweden', 'Brazil', 'Germany']
color_scale = alt.Scale(
    domain=highlight,
    range=['#1f77b4', '#d62728', '#2ca02c', '#8c564b', '#ff7f0e', '#9467bd'],
)

grey = (
    alt.Chart(stringency_monthly[~stringency_monthly['location'].isin(highlight)])
    .mark_line(opacity=0.07, color='#aab4c4', strokeWidth=0.7)
    .encode(
        x=alt.X('date:T', title='', axis=alt.Axis(format='%b %Y', tickCount={'interval': 'month', 'step': 3})),
        y=alt.Y('stringency_index:Q', title='Stringency Index', scale=alt.Scale(domain=[0, 100])),
        detail='location:N',
    )
)

highlight_sel = alt.selection_point(fields=['location'], bind='legend')
lines2 = (
    alt.Chart(stringency_monthly[stringency_monthly['location'].isin(highlight)])
    .mark_line(strokeWidth=2.5, interpolate='monotone')
    .encode(
        x='date:T', y='stringency_index:Q',
        color=alt.Color('location:N', scale=color_scale, title='Country'),
        opacity=alt.condition(highlight_sel, alt.value(1), alt.value(0.15)),
        tooltip=[
            alt.Tooltip('location:N', title='Country'),
            alt.Tooltip('date:T', title='Date', format='%b %Y'),
            alt.Tooltip('stringency_index:Q', title='Stringency Index', format='.1f'),
        ],
    )
    .add_params(highlight_sel)
)

events2 = pd.DataFrame([
    {'date': '2020-01-23', 'label': 'Wuhan Lockdown',      'y': 95},
    {'date': '2020-03-11', 'label': 'WHO Pandemic',        'y': 85},
    {'date': '2020-03-26', 'label': 'NZ Lockdown',         'y': 75},
    {'date': '2020-06-08', 'label': 'NZ Reopens',          'y': 95},
    {'date': '2021-01-20', 'label': 'US Mask Mandate',     'y': 85},
    {'date': '2021-07-01', 'label': 'Delta Wave',          'y': 75},
    {'date': '2021-12-01', 'label': 'Omicron Wave',        'y': 95},
    {'date': '2022-03-01', 'label': 'Restrictions Lifted', 'y': 85},
])
events2['date'] = pd.to_datetime(events2['date'])
ann_rules = alt.Chart(events2).mark_rule(strokeDash=[4,4], opacity=0.35, color='#555').encode(x='date:T')
ann_text  = (
    alt.Chart(events2)
    .mark_text(align='left', dx=5, dy=-5, fontSize=9, color='#555', fontStyle='italic')
    .encode(x='date:T', y='y:Q', text='label:N')
)

waves = pd.DataFrame([
    {'start': '2021-06-01', 'end': '2021-10-01'},
    {'start': '2021-11-15', 'end': '2022-02-15'},
])
waves['start'] = pd.to_datetime(waves['start'])
waves['end']   = pd.to_datetime(waves['end'])
bands2 = alt.Chart(waves).mark_rect(opacity=0.06, color='#0f3460').encode(x='start:T', x2='end:T')

chart2 = (
    (bands2 + grey + lines2 + ann_rules + ann_text)
    .properties(width=900, height=460)
    .configure_axis(grid=True, gridOpacity=0.15, labelFontSize=11, titleFontSize=12)
    .configure_view(strokeWidth=0)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)

# ── Tab 3: Policy timing vs mortality ─────────────────────────────────────────

first_death = (
    owid[owid['total_deaths'] > 0].groupby('location')['date']
    .min().reset_index().rename(columns={'date': 'first_death_date'})
)
strong_restriction = (
    owid[owid['stringency_index'] >= 70].groupby('location')['date']
    .min().reset_index().rename(columns={'date': 'restriction_date'})
)
country_dates = pd.merge(first_death, strong_restriction, on='location', how='inner')
country_dates['days_to_restrictions'] = (
    country_dates['restriction_date'] - country_dates['first_death_date']
).dt.days

deaths_pm = owid.groupby('location')['total_deaths_per_million'].max().reset_index()
meta = owid.groupby('location').agg(continent=('continent','first'), population=('population','first')).reset_index()
country_owid = country_dates.merge(deaths_pm, on='location').merge(meta, on='location')
country_owid = country_owid[country_owid['continent'].notna()]

label_countries = ['United States','Italy','China','India','Brazil','United Kingdom','Sweden','New Zealand']
continent_sel = alt.selection_point(fields=['continent'], bind='legend')

points3 = (
    alt.Chart(country_owid)
    .mark_circle(opacity=0.75, stroke='white', strokeWidth=0.8)
    .encode(
        x=alt.X('days_to_restrictions:Q', title='Policy Timing (days relative to first confirmed death)'),
        y=alt.Y('total_deaths_per_million:Q', title='COVID Deaths per Million'),
        color=alt.Color('continent:N', title='Continent'),
        size=alt.Size('population:Q', title='Population', scale=alt.Scale(range=[40, 2200])),
        opacity=alt.condition(continent_sel, alt.value(0.75), alt.value(0.1)),
        tooltip=[
            alt.Tooltip('location:N', title='Country'),
            alt.Tooltip('continent:N', title='Continent'),
            alt.Tooltip('days_to_restrictions:Q', title='Days to Restrictions'),
            alt.Tooltip('total_deaths_per_million:Q', title='Deaths per Million', format='.1f'),
            alt.Tooltip('population:Q', title='Population', format=','),
        ],
    )
    .add_params(continent_sel)
)
regression3 = (
    alt.Chart(country_owid)
    .transform_regression('days_to_restrictions', 'total_deaths_per_million')
    .mark_line(color='#0f3460', strokeDash=[6,4], size=2, opacity=0.6)
    .encode(x='days_to_restrictions:Q', y='total_deaths_per_million:Q')
)
labels3 = (
    alt.Chart(country_owid[country_owid['location'].isin(label_countries)])
    .mark_text(align='left', baseline='middle', dx=8, dy=-8, fontSize=10, fontWeight=500)
    .encode(x='days_to_restrictions:Q', y='total_deaths_per_million:Q',
            text='location:N', color=alt.value('#333'))
)
zero_line3 = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='#aaa', strokeDash=[4,4], size=1).encode(x='x:Q')

chart3 = (
    (points3 + regression3 + labels3 + zero_line3)
    .properties(width=900, height=500)
    .configure_axis(labelFontSize=11, titleFontSize=12, grid=True, gridOpacity=0.12)
    .configure_legend(titleFontSize=12, labelFontSize=11)
    .configure_view(strokeWidth=0)
)

# ── Tab 4: Animated Plotly choropleth ─────────────────────────────────────────

map_df = owid[
    owid['continent'].notna() & owid['iso_code'].notna() & owid['stringency_index'].notna()
].copy()
map_df = map_df[~map_df['iso_code'].str.startswith('OWID')]
map_df['month_label'] = map_df['date'].dt.to_period('M').dt.to_timestamp().dt.strftime('%Y-%m')
monthly_s4 = map_df.groupby(['iso_code','location','month_label'], as_index=False)['stringency_index'].mean()

fig4 = px.choropleth(
    monthly_s4, locations='iso_code', color='stringency_index',
    hover_name='location', animation_frame='month_label',
    color_continuous_scale='OrRd', range_color=(0, 100),
    projection='natural earth', labels={'stringency_index': 'Stringency Index'},
)
fig4.update_layout(
    height=520, margin=dict(l=0, r=0, t=10, b=0),
    paper_bgcolor='rgba(0,0,0,0)',
    geo=dict(bgcolor='rgba(0,0,0,0)', showframe=False, showcoastlines=True, coastlinecolor='#ccc'),
    coloraxis_colorbar=dict(title='Stringency', thickness=14, len=0.6),
    font=dict(family='Inter, sans-serif'),
)

# ── Tab 5: Stringency + mortality trends ──────────────────────────────────────

peer_countries = ['United States','United Kingdom','Canada','Germany','Italy','Sweden','Australia','Japan']
ts_df = owid[owid['location'].isin(peer_countries)][
    ['date','location','stringency_index','new_deaths_per_million']
].copy().rename(columns={'new_deaths_per_million': 'mortality_rate'})
ts_df = ts_df[ts_df['stringency_index'].notna() | ts_df['mortality_rate'].notna()]
ts_df['week'] = ts_df['date'].dt.to_period('W').dt.start_time
weekly_df = (
    ts_df.groupby(['week','location'], as_index=False)
    .agg(stringency_index=('stringency_index','mean'), mortality_rate=('mortality_rate','mean'))
)

sel5   = alt.selection_point(fields=['location'], bind='legend')
color5 = alt.Color('location:N', title='Country', scale=alt.Scale(scheme='tableau10'))

s_chart5 = (
    alt.Chart(weekly_df).mark_line(size=2, interpolate='monotone')
    .encode(
        x=alt.X('week:T', title=''),
        y=alt.Y('stringency_index:Q', title='Stringency Index', scale=alt.Scale(domain=[0,100])),
        color=color5,
        opacity=alt.condition(sel5, alt.value(1), alt.value(0.1)),
        tooltip=[alt.Tooltip('week:T',title='Week'), alt.Tooltip('location:N',title='Country'),
                 alt.Tooltip('stringency_index:Q',title='Stringency',format='.1f')],
    )
    .add_params(sel5)
    .properties(width=900, height=250, title=alt.TitleParams('Policy Stringency Over Time', fontSize=13, fontWeight=600))
)
m_chart5 = (
    alt.Chart(weekly_df).mark_line(size=2, interpolate='monotone')
    .encode(
        x=alt.X('week:T', title='Date'),
        y=alt.Y('mortality_rate:Q', title='New Deaths per Million'),
        color=color5,
        opacity=alt.condition(sel5, alt.value(1), alt.value(0.1)),
        tooltip=[alt.Tooltip('week:T',title='Week'), alt.Tooltip('location:N',title='Country'),
                 alt.Tooltip('mortality_rate:Q',title='Deaths per Million',format='.2f')],
    )
    .properties(width=900, height=250, title=alt.TitleParams('Mortality Trends Over Time', fontSize=13, fontWeight=600))
)
chart5 = (
    alt.vconcat(s_chart5, m_chart5).resolve_scale(color='shared')
    .configure_axis(grid=True, gridOpacity=0.12, labelFontSize=11, titleFontSize=12)
    .configure_view(strokeWidth=0)
    .configure_legend(titleFontSize=12, labelFontSize=11, orient='right')
)

# ── Assemble Tab widget ────────────────────────────────────────────────────────

tabs_info = [
    ('\U0001f5fa\ufe0f  Deaths Map',
     'Cumulative COVID-19 deaths per million population (log scale).', chart1),
    ('\U0001f4c8  Stringency Over Time',
     'Monthly Oxford Stringency Index 2020\u20132022. Click legend to highlight countries.', chart2),
    ('\u23f1\ufe0f  Policy Timing vs Mortality',
     'Did acting faster reduce deaths? Bubble size = population. Click legend to filter by continent.', chart3),
    ('\U0001f3ac  Animated Stringency Map',
     'How policy stringency evolved globally month by month. Press play or drag the slider.', fig4),
    ('\U0001f4c9  Policy & Mortality Trends',
     'Weekly stringency and mortality for 8 peer nations. Click legend to isolate a country.', chart5),
]

out_widgets = []
for title, desc, chart in tabs_info:
    out = widgets.Output()
    with out:
        display(HTML(f'<div class="section-header">{title.strip()}</div><div class="section-desc">{desc}</div>'))
        if hasattr(chart, 'show'):   # plotly
            chart.show()
        else:                        # altair
            display(chart)
    out_widgets.append(out)

tab = widgets.Tab(children=out_widgets)
for i, (title, _, _) in enumerate(tabs_info):
    tab.set_title(i, title)

display(tab)